#### 02 - Cleaning & Feature Engineering
Load the output from notebook 01, create color index features, and prepare
datasets for classification and galaxy redshift regression.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BASE_PATH = PROCESSED_DIR / "sdss_base_150k.csv"
BASE_PATH

WindowsPath('c:/uni/seriousism/Redshifted/data/processed/sdss_base_150k.csv')

In [3]:
df = pd.read_csv(BASE_PATH)
df.shape

(150000, 14)

In [4]:
df["objID"] = df["objID"].astype("string")
df["specObjID"] = df["specObjID"].astype("string")
df["object_class"] = df["object_class"].astype("string")

df.dtypes

objID            string
specObjID        string
ra              float64
dec             float64
object_class     string
z               float64
zErr            float64
zWarning          int64
dered_u         float64
dered_g         float64
dered_r         float64
dered_i         float64
dered_z         float64
source_file         str
dtype: object

In [5]:
before = df.shape[0]
df = df.drop_duplicates(subset=["specObjID"])
print(f"Dropped {before - df.shape[0]} rows")

Dropped 0 rows


In [8]:
mag_cols = ["dered_u", "dered_g", "dered_r", "dered_i", "dered_z"]
df = df[(df[mag_cols] >= 0).all(axis=1) & (df[mag_cols] <= 30).all(axis=1)]
df.shape

(150000, 14)

Color indices (differences between filters) are more informative than raw
magnitude for distinguishing object types, since raw magnitude is affected by
distance and intrinsic brightness, while the difference more purely reflects
spectral composition.

In [11]:
df["u_g"] = df["dered_u"] - df["dered_g"]
df["g_r"] = df["dered_g"] - df["dered_r"]
df["r_i"] = df["dered_r"] - df["dered_i"]
df["i_z"] = df["dered_i"] - df["dered_z"]

df[["u_g", "g_r", "r_i", "i_z"]].describe()

,u_g,g_r,r_i,i_z
count,150000.000000,150000.000000,150000.000000,150000.000000
mean,1.138156,0.652770,0.412630,0.239089
std,1.019862,0.681519,0.561552,0.479729
min,-11.917950,-11.132460,-8.896440,-12.387870
25%,0.301340,0.187398,0.103700,0.028847
50%,1.051400,0.480305,0.305260,0.218945
75%,1.790583,1.038870,0.496980,0.372950
max,12.547710,11.572360,14.529170,10.276500


In [13]:
df_galaxy = df[
  (df["object_class"] == "GALAXY") &
  (df["z"] > 0) &
  (df["z"] < 1)
].copy()

df_galaxy.shape

(49402, 18)

No `zErr` filter applied here yet. Noise in the `z` target is better assessed
during EDA (notebook 03) first — filtering only if the data actually shows a
need for it, so the decision is grounded in what the data looks like rather
than an upfront assumption.

In [14]:
print("Missing values (df):")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nMissing values (df_galaxy):")
print(df_galaxy.isna().sum()[df_galaxy.isna().sum() > 0])

Missing values (df):
Series([], dtype: int64)

Missing values (df_galaxy):
Series([], dtype: int64)


In [16]:
df.to_csv(PROCESSED_DIR / "sdss_clean.csv", index=False)
df_galaxy.to_csv(PROCESSED_DIR / "sdss_galaxy_redshift.csv", index=False)

print(f"df ({df.shape}) -> sdss_clean.csv")
print(f"df_galaxy ({df_galaxy.shape}) -> sdss_galaxy_redshift.csv")

df ((150000, 18)) -> sdss_clean.csv
df_galaxy ((49402, 18)) -> sdss_galaxy_redshift.csv


#### Cleaning & Feature Engineering Summary

Added 4 color index features (u_g, g_r, r_i, i_z) from the 5 dereddened
magnitudes. Full dataset saved as `sdss_clean.csv` for classification;
galaxy-only subset (0 < z < 1) saved as `sdss_galaxy_redshift.csv` for
redshift regression.

zErr filtering deferred to EDA — want to see the noise before deciding
whether to cut it.